# Experiment 10 - Part 3: The Production Service: Cache, Logs, Traces and a Load Test

**Course:** HAAI302COM - Artificial Intelligence Laboratory

We now assemble everything into one service, `app.py`:

```
client --> [request id, timing] --> [API key + rate limit (Redis)] --> /chat/stream
                                                                          |
                                      semantic cache lookup (embed + Redis)
                                         hit  -> stream the stored answer
                                         miss -> stream Gemini's answer, then store it
```

On top of the cache from notebook 02 we add what a real deployment needs:

- **request logging**: one JSON line per request with an id, status and latency;
- **tracing**: a tree of timed *spans* for each request (lookup, embedding, model call, store), written to a file and, when you have an account, also sent to **LangSmith**;
- **authentication and rate limiting** with an API key header and a Redis counter;
- a **/metrics** endpoint;

and then we **load-test** the service with and without the cache.

Run notebooks 01 and 02 first: this one needs `semcache.py`, `traffic.py` and `chosen_threshold.txt` from notebook 02, and the Redis container.

## 0. Setup

In [1]:
import os, time, json, asyncio, warnings, glob
warnings.filterwarnings("ignore")
import httpx, redis
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import svc_utils
from traffic import GROUPS

load_dotenv(find_dotenv(usecwd=True))
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 170)
THRESHOLD = float(open("chosen_threshold.txt").read())
r = redis.Redis(host="127.0.0.1", port=6379)
for f in ["requests.jsonl", "traces.jsonl"]:
    if os.path.exists(f): os.remove(f)
for k in r.scan_iter("svc:*"): r.delete(k)
print("redis:", r.ping(), "| cache threshold from notebook 02:", THRESHOLD)

redis: True | cache threshold from notebook 02: 0.85


## 1. The service, part A: configuration, logging and tracing

**Logging.** `log_event` appends one JSON object per line to `requests.jsonl` ("JSON lines"). Structured logs can be searched and loaded into a table; free text cannot.

**Tracing.** A *trace* is the story of one request; it is made of *spans*, each with a name, a start, a duration and a parent. `Tracer.span` is a context manager that times a block and records its parent, so the spans form a tree. We write them to `traces.jsonl`.

**LangSmith.** LangSmith is LangChain's hosted tracing service. Its `@traceable` decorator records a function call (inputs, outputs, time, errors) and sends it to your project. We apply it to the model call, but only when `LANGSMITH_TRACING=true` is set, so that the service runs the same without an account.

In [2]:
%%writefile app.py
import os, json, time, uuid, asyncio
from collections import deque
from contextlib import asynccontextmanager, contextmanager
import numpy as np
import redis
from fastapi import FastAPI, Request, Header, HTTPException, Depends
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv, find_dotenv
from semcache import SemanticCache

load_dotenv(find_dotenv(usecwd=True))
MODEL = "gemini-2.5-flash-lite"
API_KEY = os.environ.get("SERVICE_API_KEY", "dev-key")
THRESHOLD = float(os.environ.get("CACHE_THRESHOLD", "0.8"))
RATE_LIMIT = int(os.environ.get("RATE_LIMIT_PER_MIN", "120"))
MOCK = os.environ.get("MOCK_LLM") == "1"                      # canned answers: lets you run the service without an API key
SYSTEM = "Answer in at most three short sentences."

if os.environ.get("LANGSMITH_TRACING") == "true":             # send traces to LangSmith when an account is configured
    from langsmith import traceable
else:
    def traceable(**kw):
        return lambda f: f


def log_event(**kw):
    with open("requests.jsonl", "a") as f:
        f.write(json.dumps({"ts": round(time.time(), 3), **kw}) + "\n")


class Tracer:
    # records a tree of timed spans for one request in traces.jsonl
    def __init__(self, trace_id):
        self.id, self.root, self.stack, self.t0 = trace_id, uuid.uuid4().hex[:8], [], time.time()

    def _write(self, sid, parent, name, t, attrs):
        with open("traces.jsonl", "a") as f:
            f.write(json.dumps({"trace_id": self.id, "span_id": sid, "parent_id": parent, "name": name,
                                "start_ms": round((t - self.t0) * 1000, 2), "ms": round((time.time() - t) * 1000, 2), "attrs": attrs}) + "\n")

    @contextmanager
    def span(self, name, **attrs):
        sid, parent, t = uuid.uuid4().hex[:8], (self.stack[-1] if self.stack else self.root), time.time()
        self.stack.append(sid)
        try:
            yield attrs
        finally:
            self.stack.pop()
            self._write(sid, parent, name, t, attrs)

    def finish(self, **attrs):
        self._write(self.root, None, "request", self.t0, attrs)

Overwriting app.py


## 2. The service, part B: models, startup, request-id middleware, key and rate limit

- **Lifespan**: code that runs once when the server starts. We load the embedding model (slow, so never per request), connect to Redis and create the Gemini client and the `SemanticCache`. They are stored on `app.state`.
- **Middleware**: a function that wraps every request. Ours gives each request an **id** (or keeps the one the client sent in `x-request-id`), returns it in the `X-Request-ID` header and logs the status and time until the response started.
- **`guard` dependency**: checks the `x-api-key` header (401 if wrong) and counts requests per key per minute in Redis with `incr` (429 if over the limit). `Depends(guard)` runs it before the endpoint.

In [3]:
%%writefile -a app.py


class ChatRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=2000)
    max_tokens: int = Field(150, ge=1, le=600)
    temperature: float = Field(0.0, ge=0, le=1)
    use_cache: bool = True


@asynccontextmanager
async def lifespan(app):
    app.state.r = redis.Redis(host="127.0.0.1", port=6379)
    app.state.cache = SemanticCache(app.state.r, SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"),
                                    threshold=THRESHOLD, ttl=3600, namespace="svc:cache")
    app.state.llm = None if MOCK else AsyncOpenAI(api_key=os.environ["GEMINI_API_KEY"],
                                                  base_url="https://generativelanguage.googleapis.com/v1beta/openai/", max_retries=2)
    app.state.lat = {"hit": deque(maxlen=2000), "miss": deque(maxlen=2000), "bypass": deque(maxlen=2000)}
    app.state.errors = 0
    yield


app = FastAPI(title="LLM service with semantic cache", lifespan=lifespan)


@app.middleware("http")
async def request_id_and_log(request: Request, call_next):
    rid = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]
    request.state.request_id, t0 = rid, time.time()
    try:
        response = await call_next(request)
    except Exception as e:
        log_event(kind="http", request_id=rid, path=request.url.path, status=500, error=type(e).__name__)
        return JSONResponse({"detail": "internal error", "request_id": rid}, status_code=500, headers={"X-Request-ID": rid})
    response.headers["X-Request-ID"] = rid
    log_event(kind="http", request_id=rid, method=request.method, path=request.url.path, status=response.status_code,
              ttfb_ms=round((time.time() - t0) * 1000, 1))
    return response


async def guard(request: Request, x_api_key: str = Header(default="")):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="invalid or missing API key")
    key = f"svc:rl:{x_api_key}:{int(time.time() // 60)}"
    n = await run_in_threadpool(lambda: (request.app.state.r.incr(key), request.app.state.r.expire(key, 90))[0])
    if n > RATE_LIMIT:
        raise HTTPException(status_code=429, detail="rate limit exceeded", headers={"Retry-After": "30"})

Appending to app.py


## 3. The service, part C: the streaming endpoint

The order of work in `/chat/stream`:

1. (If `use_cache`) embed the prompt and look it up. Both are CPU work, so they run in `run_in_threadpool` to keep the event loop free (the lesson of notebook 01).
2. Return a `StreamingResponse` with headers `X-Cache` (hit, miss or bypass) and `X-Cache-Score`.
3. The first event is `meta` (request id, cache status, similarity). Then either the cached answer in small pieces, or Gemini's tokens as they arrive. After a successful miss the answer is stored in the cache.
4. In `finally`, whatever happens (even if the client disconnects), the root span, the log line and the latency metrics are written.

Mock mode (`MOCK_LLM=1`) replaces Gemini with a canned answer, so the cache, logging and load test can be tried without a key.

In [4]:
%%writefile -a app.py


@traceable(name="gemini_stream", run_type="llm")
async def llm_stream(llm, prompt, max_tokens, temperature):
    if MOCK:
        for w in f"[mock answer] You asked: {prompt[:80]}".split():
            await asyncio.sleep(0.05)
            yield w + " "
        return
    stream = await llm.chat.completions.create(model=MODEL, max_tokens=max_tokens, temperature=temperature, stream=True,
                                               messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}])
    async for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else ""
        if piece:
            yield piece


def sse(obj):
    return f"data: {json.dumps(obj)}\n\n"


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest, request: Request, _: None = Depends(guard)):
    rid, st = request.state.request_id, request.app.state
    tr, t0, answer, score, vec, status = Tracer(rid), time.time(), None, 0.0, None, "bypass"
    if req.use_cache:
        with tr.span("cache.lookup") as a:
            with tr.span("embed"):
                vec = await run_in_threadpool(st.cache.embed, req.prompt)
            answer, score, _ = await run_in_threadpool(st.cache.lookup, req.prompt, vec)
            a["hit"], a["score"] = answer is not None, round(score, 3)
        status = "hit" if answer is not None else "miss"

    async def events():
        full, first, err = [], None, None
        try:
            yield sse({"meta": {"request_id": rid, "cache": status, "score": round(score, 3)}})
            if answer is not None:
                words = answer.split(" ")
                for i in range(0, len(words), 4):                     # replay the stored answer in small pieces
                    first = first or time.time() - t0
                    yield sse({"token": " ".join(words[i:i + 4]) + " "})
            else:
                with tr.span("llm.stream", model="mock" if MOCK else MODEL) as a:
                    async for piece in llm_stream(st.llm, req.prompt, req.max_tokens, req.temperature):
                        first = first or time.time() - t0
                        full.append(piece)
                        yield sse({"token": piece})
                    a["chars"] = len("".join(full))
                if req.use_cache and full:
                    with tr.span("cache.store"):
                        await run_in_threadpool(st.cache.store, req.prompt, "".join(full).strip(), vec)
            yield "data: [DONE]\n\n"
        except Exception as e:
            err = type(e).__name__
            st.errors += 1
            yield sse({"error": err})
        finally:
            total = (time.time() - t0) * 1000
            st.lat[status].append(total)
            tr.finish(cache=status, error=err)
            log_event(kind="chat", request_id=rid, cache=status, score=round(score, 3), ttft_ms=round((first or 0) * 1000, 1),
                      total_ms=round(total, 1), error=err, prompt_chars=len(req.prompt))

    return StreamingResponse(events(), media_type="text/event-stream", headers={"X-Cache": status, "X-Cache-Score": f"{score:.3f}"})

Appending to app.py


## 4. The service, part D: health, metrics and cache administration

`/metrics` summarises what the process has seen: counts, hit rate, and the 50th and 95th percentile latency (**p50**: half of requests were faster; **p95**: only one in twenty was slower) for each cache status. Percentiles, not averages, are what you put in a service-level objective, because averages hide the slow tail.

In [5]:
%%writefile -a app.py


@app.get("/health")
async def health():
    return {"status": "ok", "model": "mock" if MOCK else MODEL, "threshold": THRESHOLD}


@app.get("/metrics")
async def metrics(request: Request, _: None = Depends(guard)):
    st = request.app.state
    pct = lambda d, p: round(float(np.percentile(d, p)), 1) if len(d) else None
    n = {k: len(v) for k, v in st.lat.items()}
    answered = n["hit"] + n["miss"]
    return {"requests": n, "errors": st.errors, "hit_rate": round(n["hit"] / answered, 3) if answered else 0.0,
            "p50_ms": {k: pct(v, 50) for k, v in st.lat.items()}, "p95_ms": {k: pct(v, 95) for k, v in st.lat.items()},
            "cache": st.cache.stats()}


@app.post("/cache/clear")
async def cache_clear(request: Request, _: None = Depends(guard)):
    await run_in_threadpool(request.app.state.cache.clear)
    return {"cleared": True}

Appending to app.py


In [6]:
server = svc_utils.start_server("app", 8102, env={"CACHE_THRESHOLD": str(THRESHOLD), "RATE_LIMIT_PER_MIN": "40"})
URL, H = "http://127.0.0.1:8102", {"x-api-key": "dev-key"}
print(httpx.get(URL + "/health").json())

{'status': 'ok', 'model': 'gemini-2.5-flash-lite', 'threshold': 0.85}


## 5. Security checks: API key and rate limit

Without the key the service answers 401; with a wrong key also 401. We set the limit to 40 requests per minute for this run, then send 45 quick requests with the right key to a cheap endpoint: the first 40 pass and the rest get 429 with a `Retry-After` header. The rate limit protects both your bill and other users.

In [7]:
print("no key   :", httpx.get(URL + "/metrics").status_code, httpx.get(URL + "/metrics").json())
print("wrong key:", httpx.get(URL + "/metrics", headers={"x-api-key": "nope"}).status_code)
codes = [httpx.get(URL + "/metrics", headers=H).status_code for _ in range(45)]      # only requests with the right key are counted
resp = httpx.get(URL + "/metrics", headers=H)
print("45 requests ->", {c: codes.count(c) for c in sorted(set(codes))}, "| Retry-After:", resp.headers.get("retry-after"))

no key   : 401 {'detail': 'invalid or missing API key'}
wrong key: 401


45 requests -> {200: 40, 429: 5} | Retry-After: 30


**What to notice:** a missing key and a wrong key both get 401 with a clear message. With the limit set to 40 per minute, 45 quick requests with the right key gave 40 successes and 5 rejections (429) with a Retry-After of 30 seconds. Only requests with a valid key are counted, so an attacker without a key cannot use up a real user's allowance.

## 6. A miss, then a paraphrase: the cache in action

The rate-limit counter lives in Redis under the current minute, so we start a new server with a higher limit for the rest of the notebook. Then: ask a question (miss, the model answers), ask the same thing in other words (hit), and ask a look-alike that means something else (should be a miss). The helper prints the response headers and the `meta` event.

In [8]:
svc_utils.stop_server(server)
for k in r.scan_iter("svc:*"): r.delete(k)
server = svc_utils.start_server("app", 8102, env={"CACHE_THRESHOLD": str(THRESHOLD), "RATE_LIMIT_PER_MIN": "1000"})

def ask(prompt, use_cache=True):
    t0, first, toks, meta = time.time(), None, [], None
    with httpx.stream("POST", URL + "/chat/stream", headers=H, json={"prompt": prompt, "use_cache": use_cache}, timeout=60) as resp:
        for line in resp.iter_lines():
            if not line.startswith("data: ") or line[6:] == "[DONE]": continue
            ev = json.loads(line[6:])
            if "meta" in ev: meta = ev["meta"]
            elif "token" in ev:
                first = first or time.time() - t0; toks.append(ev["token"])
    return dict(cache=resp.headers["x-cache"], score=float(resp.headers["x-cache-score"]), ttft_s=round(first or 0, 2),
                total_s=round(time.time() - t0, 2), request_id=resp.headers["x-request-id"], answer="".join(toks).strip())

In [9]:
rows = [dict(prompt=p, **ask(p)) for p in ["What is the capital of France?",                 # first time: miss
                                          "Which city is the capital of France?",           # paraphrase: hit
                                          "What is the capital of Germany?",                # look-alike: should be a miss
                                          "Which city is the capital of France?"]]          # repeat: hit
pd.DataFrame(rows)[["prompt", "cache", "score", "ttft_s", "total_s", "answer"]]

,prompt,cache,score,ttft_s,total_s,answer
0,What is the capital of France?,miss,0.000,1.69,1.79,The capital of France is Paris. It is known for its iconic landmarks like th...
1,Which city is the capital of France?,hit,0.942,0.04,0.04,The capital of France is Paris. It is known for its iconic landmarks like th...
2,What is the capital of Germany?,miss,0.663,0.96,1.12,The capital of Germany is Berlin. It is the largest city in Germany and a ma...
3,Which city is the capital of France?,hit,0.942,0.04,0.04,The capital of France is Paris. It is known for its iconic landmarks like th...


**What to notice:** the first question is a miss: the model answers and the first token arrives after about 1.7 s. The paraphrase 'Which city is the capital of France?' is a hit with similarity 0.942, and its first token arrives after 0.04 s with the same answer, about forty times faster. The look-alike 'capital of Germany' scores only 0.663, so it correctly misses and the model answers Berlin in about 1.1 s. The repeated paraphrase is a hit again.

## 7. Reading the logs

Because every line of `requests.jsonl` is JSON, `pd.read_json(lines=True)` turns it into a table. There are two kinds of record: `http` (written by the middleware as soon as the response started) and `chat` (written when the stream ended). They share the `request_id`, which is how you join them, and how a user can quote a failing request to support.

In [10]:
logs = pd.read_json("requests.jsonl", lines=True)
print("record kinds:", logs.kind.value_counts().to_dict())
logs[logs.kind == "chat"][["request_id", "cache", "score", "ttft_ms", "total_ms", "error"]].tail(6)

record kinds: {'http': 56, 'chat': 4}


,request_id,cache,score,ttft_ms,total_ms,error
53,7dd12dfb6459,miss,0.000,1669.1,1768.6,NaN
55,3cd97bf201ab,hit,0.942,19.3,20.2,NaN
57,510a9f6a93ec,miss,0.663,946.1,1104.5,NaN
59,0c913c538739,hit,0.942,19.4,21.6,NaN


**What to notice:** there are 60 records: 56 http records (every call, including the rate-limit test) and 4 chat records. A hit takes about 20 ms on the server and a miss 1.1 to 1.8 s; the client measured 40 ms for a hit because it also includes the network and the client library. The request_id in each record is the same as the X-Request-ID header the client received, so a user who reports a problem can give you one string that finds the whole story.

## 8. Reading the traces

Each request wrote a tree of spans. We print the trace of the first request (the miss) and of the second (the hit) as indented trees, with the time of each span. The root span `request` covers the whole stream; children are nested by `parent_id`.

In [11]:
spans = pd.read_json("traces.jsonl", lines=True)
def show(trace_id):
    s = spans[spans.trace_id == trace_id].sort_values("start_ms")
    depth = {None: -1}
    for _, row in s.iterrows():
        depth[row.span_id] = depth.get(row.parent_id, -1) + 1
        print("  " * depth[row.span_id] + f"{row['name']:<14} {row.ms:8.1f} ms   {row['attrs']}")
chat = logs[logs.kind == "chat"].reset_index(drop=True)
for i in (0, 1):
    print(f"--- request {i + 1} ({chat.cache[i]}) ---"); show(chat.request_id[i])

--- request 1 (miss) ---
request          1768.7 ms   {'cache': 'miss', 'error': None}
  cache.lookup       48.7 ms   {'hit': False, 'score': 0.0}
    embed              46.3 ms   {}
  llm.stream       1714.3 ms   {'model': 'gemini-2.5-flash-lite', 'chars': 177}
  cache.store         3.8 ms   {}
--- request 2 (hit) ---
request            20.3 ms   {'cache': 'hit', 'error': None}
  cache.lookup       18.3 ms   {'hit': True, 'score': 0.9420000000000001}
    embed              15.8 ms   {}


**What to notice:** on the miss (1.77 s) the model call is 1.71 s, which is 97 percent of the time; the cache lookup costs 49 ms (46 ms of it embedding the question) and storing the answer 3.8 ms. So the cache adds about 50 ms to a miss, around 3 percent: cheap insurance. On the hit the whole request takes 20 ms and nearly all of it is the lookup, with the embedding at 16 ms. The embedding step is the main price of the cache, and that is why the embedding model runs in a thread pool and is loaded only once at startup.

In [12]:
spans.groupby("name").ms.agg(["count", "mean", "max"]).round(1).sort_values("mean", ascending=False)

,count,mean,max
name,,,
llm.stream,2,1399.0,1714.3
request,4,728.8,1768.7
cache.lookup,4,25.1,48.7
embed,4,22.8,46.3
cache.store,2,3.6,3.8


**What to notice:** over the four requests, the model call averages 1.4 s, the lookup 25 ms and the store 3.6 ms. The 'request' row is the parent of the others, so its time includes theirs and the rows must not be added together. Grouping spans by name like this is how you find which stage to optimise: here only the model call matters.

## 9. Load test: cache off against cache on

A single request tells us nothing about a service under traffic. We send **32 requests**: 8 questions, each in its 4 paraphrases, shuffled, from 4 concurrent clients (`asyncio` with `httpx.AsyncClient` and a semaphore). We run it twice on identical traffic: with `use_cache=false` (every request calls Gemini) and with `use_cache=true` (cache emptied first). We record per request the time to first token and the total time.

In [13]:
import random
names = list(GROUPS)[:8]
workload = [(g, q) for g in names for q in GROUPS[g]]
random.Random(1).shuffle(workload)

async def one(client, sem, g, q, use_cache):
    async with sem:
        t0, first, status = time.time(), None, None
        async with client.stream("POST", "/chat/stream", headers=H, json={"prompt": q, "use_cache": use_cache}) as resp:
            status = resp.headers["x-cache"]
            async for line in resp.aiter_lines():
                if line.startswith("data: ") and first is None and "token" in line:
                    first = time.time() - t0
        return dict(group=g, cache=status, ttft=first or 0, total=time.time() - t0)

async def load(use_cache, concurrency=4):
    sem = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(base_url=URL, timeout=90) as client:
        t0 = time.time()
        rows = await asyncio.gather(*[one(client, sem, g, q, use_cache) for g, q in workload])
        return pd.DataFrame(rows), time.time() - t0

httpx.post(URL + "/cache/clear", headers=H)
off, wall_off = await load(False)
httpx.post(URL + "/cache/clear", headers=H)
on, wall_on = await load(True)
print("cache off: wall", round(wall_off, 1), "s | cache on: wall", round(wall_on, 1), "s")

cache off: wall 9.0 s | cache on: wall 5.8 s


In [14]:
def summ(df, wall, name):
    return dict(run=name, requests=len(df), model_calls=int((df.cache != "hit").sum()), hit_rate=round((df.cache == "hit").mean(), 2),
                p50_total_s=round(df.total.median(), 2), p95_total_s=round(df.total.quantile(.95), 2), p50_ttft_s=round(df.ttft.median(), 2),
                wall_s=round(wall, 1), requests_per_s=round(len(df) / wall, 1))
pd.DataFrame([summ(off, wall_off, "cache off"), summ(on, wall_on, "cache on")]).set_index("run").T

run,cache off,cache on
requests,32.00,32.00
model_calls,32.00,19.00
hit_rate,0.00,0.41
p50_total_s,1.10,1.06
p95_total_s,1.25,1.23
p50_ttft_s,0.94,0.90
wall_s,9.00,5.80
requests_per_s,3.50,5.50


**What to notice:** with the cache on, the same 32 requests needed 19 model calls instead of 32 (13 hits, a hit rate of 0.41), finished in 5.8 s instead of 9.0 s, and the service handled 5.5 requests per second instead of 3.5. But the median and the 95th percentile of latency are almost unchanged (1.06 s against 1.10 s), because 59 percent of requests were still misses, so the median request is still a model call. A cache lowers cost and raises capacity at once; it lowers the typical latency only when the hit rate is above one half. The best possible here was 24 hits of 32 (75 percent); we got 13, which is 54 percent of that: the threshold of 0.85 gives up about 40 percent of the paraphrase matches, as the sweep in notebook 02 predicted.

In [15]:
on.groupby("cache")[["ttft", "total"]].agg(["count", "median"]).round(3)

ttft        total       
      count median count median
cache                          
hit      13  0.032    13  0.035
miss     19  0.944    19  1.139

**What to notice:** split by outcome, a hit has a median total time of 35 ms and a miss 1.14 s, about 33 times faster. The speed-up is large for the requests that hit, which is why the average improves even when the median does not.

In [16]:
print(json.dumps(httpx.get(URL + "/metrics", headers=H).json(), indent=1))

{
 "requests": {
  "hit": 15,
  "miss": 21,
  "bypass": 32
 },
 "errors": 0,
 "hit_rate": 0.417,
 "p50_ms": {
  "hit": 27.3,
  "miss": 1129.3,
  "bypass": 1088.3
 },
 "p95_ms": {
  "hit": 41.9,
  "miss": 1254.6,
  "bypass": 1240.1
 },
 "cache": {
  "hits": 13,
  "misses": 19,
  "hit_rate": 0.406,
  "entries": 19
 }
}


**What to notice:** the server counted 15 hits, 21 misses, 32 bypassed requests (cache off) and no errors. Its hit rate of 0.417 includes the four requests from section 6, while the Redis counters (13 hits and 19 misses, 0.406, with 19 entries stored) cover only the load-test run after the cache was cleared. The percentiles tell the story again: hits 27 ms median and 42 ms at the 95th percentile, misses 1.13 s and 1.25 s. These numbers live in the process and reset when it restarts; a production service would export them to a system such as Prometheus.

## 10. LangSmith tracing

Our own spans live in a local file. **LangSmith** gives the same tree in a web UI, with the prompt and answer of every model call, token counts, errors and search. To switch it on you need a free account at smith.langchain.com and an API key, then:

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=<your key>
LANGSMITH_PROJECT=haai302-exp10
```

in `.env` (never commit it). The service already decorates the model call with `@traceable`; nothing else changes. The cell below runs only if the key is present: it restarts the server with tracing enabled and sends one request.

In [17]:
if os.environ.get("LANGSMITH_API_KEY"):
    svc_utils.stop_server(server)
    server = svc_utils.start_server("app", 8102, env={"CACHE_THRESHOLD": str(THRESHOLD), "LANGSMITH_TRACING": "true", "RATE_LIMIT_PER_MIN": "1000"})
    print(ask("Explain what a span is in tracing, in two sentences.", use_cache=False))
    print("open smith.langchain.com, project", os.environ.get("LANGSMITH_PROJECT", "default"), "to see the trace")
else:
    print("LANGSMITH_API_KEY not set: skipping the LangSmith demo (the local traces above show the same structure).")

LANGSMITH_API_KEY not set: skipping the LangSmith demo (the local traces above show the same structure).


In [18]:
svc_utils.stop_server(server)
for k in r.scan_iter("svc:*"): r.delete(k)
print("server stopped, svc keys removed; lines in server log:", len(open("server_8102.log").read().splitlines()))

server stopped, svc keys removed; lines in server log: 6


## 11. Deployment notes and recap

To run this as a real service: a `Dockerfile` that installs the requirements and starts `uvicorn app:app --host 0.0.0.0 --port 8000 --workers 2`; Redis as a managed service (with persistence if the cache must survive restarts); the API key and the Gemini key in environment variables, never in the image; the `/health` endpoint wired to the platform's health check; logs collected from standard output instead of local files; and the embedding model baked into the image so that startup does not download it.

- A service is a typed contract (Pydantic), a streaming transport (SSE) and non-blocking code.
- A semantic cache saves model calls and latency; its threshold decides how many wrong answers it will return.
- Logs say *what* happened, traces say *where the time went*, metrics say *how often and how slow*; use percentiles, not averages.
- Authentication and rate limits are part of the service, not an afterthought.

**Try yourself:** (1) run the service with `MOCK_LLM=1` and no API key; (2) lower the cache threshold to 0.6 and watch the wrong answers appear in section 6; (3) raise the load-test concurrency to 16 and see which latency percentile moves first.

This completes Experiment 10 and the ten practicals.